# LUAD matched UMAP, spatial regions, and C5/C6 program enrichment

This notebook regenerates the requested figure panels from the current slide_scaled tile table: UMAP by sample/region feature, four-slide spatial key-region panel, the original C5/C6 program enrichment plot, and two 4 x 8 alternative C5/C6 program enrichment layouts.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path('/data/hulei/OmicOS')
OUT = ROOT / 'OmicOS_8'
sys.path.insert(0, str(OUT))

from scripts.plot_luad_matched_umap_spatial_regions import (
    DEFAULT_TILE_TABLE,
    FIG_DIR,
    assign_key_region_feature,
    plot_c5_c6_program_enrichment,
    plot_c5_c6_program_enrichment_dot_heatmap,
    plot_c5_c6_program_enrichment_stacked,
    plot_spatial_key_regions,
    plot_umap_sample_vs_regions,
    validate_tile_table,
)

import omicverse as ov
import scanpy as sc
import matplotlib.pyplot as plt
ov.plot_set(font_path='Arial')
sc.settings.set_figure_params(dpi=300, vector_friendly=True)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


/opt/miniforge/envs/omicverse_working/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/miniforge/envs/omicverse_working/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/miniforge/envs/omicverse_working/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


/opt/miniforge/envs/omicverse_working/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


/opt/miniforge/envs/omicverse_working/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(
/opt/miniforge/envs/omicverse_working/lib/python3.11/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


🔬 Starting plot initialization...
Using already downloaded Arial font from: /tmp/omicverse_arial.ttf


Registered as: Arial
🧬 Detecting GPU devices…
✅ NVIDIA CUDA GPUs detected: 8
    • [CUDA 0] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9
    • [CUDA 1] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9
    • [CUDA 2] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9
    • [CUDA 3] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9
    • [CUDA 4] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9
    • [CUDA 5] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9
    • [CUDA 6] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9
    • [CUDA 7] NVIDIA GeForce RTX 4090 D
      Memory: 23.5 GB | Compute: 8.9

   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                           

In [2]:
tile_df = pd.read_csv(DEFAULT_TILE_TABLE)
validate_tile_table(tile_df)
if 'integration_method' in tile_df.columns and set(tile_df['integration_method'].astype(str)) != {'slide_scaled'}:
    raise ValueError('this notebook expects the primary slide_scaled tile table')

tile_df.assign(region_feature=tile_df['region_label'].map(assign_key_region_feature))['region_feature'].value_counts()


region_feature
Other regions                   7235
Shared C6: low-risk enriched    1334
Mixed C5: high-risk enriched     729
Name: count, dtype: int64

## UMAP and spatial region panels

In [3]:
umap_base = FIG_DIR / 'luad_matched_2high_2low_umap_sample_vs_key_regions_omicos8'
spatial_base = FIG_DIR / 'luad_matched_2high_2low_spatial_key_regions_panel_omicos8'

plot_umap_sample_vs_regions(tile_df, umap_base)
plot_spatial_key_regions(tile_df, spatial_base, max_dim=2048)

for path in [
    umap_base.with_suffix('.png'),
    umap_base.with_suffix('.pdf'),
    spatial_base.with_suffix('.png'),
    spatial_base.with_suffix('.pdf'),
]:
    print(path)


/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_umap_sample_vs_key_regions_omicos8.png
/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_umap_sample_vs_key_regions_omicos8.pdf
/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_spatial_key_regions_panel_omicos8.png
/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_spatial_key_regions_panel_omicos8.pdf


## Original C5/C6 program enrichment plot

In [4]:
program_base = FIG_DIR / 'luad_matched_2high_2low_c5_c6_gene_program_enrichment_omicos8'
program_enrichment = plot_c5_c6_program_enrichment(tile_df, program_base)

for path in [program_base.with_suffix('.png'), program_base.with_suffix('.pdf')]:
    print(path)


/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_c5_c6_gene_program_enrichment_omicos8.png
/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_c5_c6_gene_program_enrichment_omicos8.pdf


## 4 x 8 C5/C6 program enrichment alternatives

In [5]:
stacked_base = FIG_DIR / 'luad_matched_2high_2low_c5_c6_gene_program_enrichment_stacked_4x8_omicos8'
dot_base = FIG_DIR / 'luad_matched_2high_2low_c5_c6_gene_program_enrichment_dot_heatmap_4x8_omicos8'

stacked_enrichment = plot_c5_c6_program_enrichment_stacked(tile_df, stacked_base)
dot_enrichment = plot_c5_c6_program_enrichment_dot_heatmap(tile_df, dot_base)

for path in [
    stacked_base.with_suffix('.png'),
    stacked_base.with_suffix('.pdf'),
    dot_base.with_suffix('.png'),
    dot_base.with_suffix('.pdf'),
]:
    print(path)


/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_c5_c6_gene_program_enrichment_stacked_4x8_omicos8.png
/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_c5_c6_gene_program_enrichment_stacked_4x8_omicos8.pdf
/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_c5_c6_gene_program_enrichment_dot_heatmap_4x8_omicos8.png
/data/hulei/OmicOS/OmicOS_8/outputs/figures/luad_matched_2high_2low_c5_c6_gene_program_enrichment_dot_heatmap_4x8_omicos8.pdf
